In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)
from src.d01_init_proc import align_imgs
import src.d00_utils.utilities as utils
import src.d00_utils.dirnames as dn
import src.d01_init_proc.vis_and_rescale as vr
from src.d01_init_proc import align_imgs, subtractbg as sb, timelapse_corr as tc


from bioio import BioImage
import bioio_ome_tiff
import bioio_tifffile
from bioio.writers import OmeTiffWriter

import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
input_dirpath = Path(input())

In [ ]:
actinch = 2

proc_dirpath = utils.get_proc_dirpath(input_dirpath)
output_dirpath = proc_dirpath / dn.labbkit_seg_dirname / 'caax_cell_bgsbratiocorractin_seg'
output_dirpath.mkdir(exist_ok=True)

In [ ]:
imgpaths = [path for path in input_dirpath.glob('*.ome.tif')]

for imgpath in tqdm(imgpaths):
    
    if not (output_dirpath / imgpath.name).is_file():
        
        img_file = BioImage(imgpath, open=bioio_ome_tiff.Reader)
        img = img_file.data

        # correct actin channel image
        actin = img[:, actinch, np.newaxis, :, :, :]
        actin = sb.subtract_median(actin)
        actin = tc.timelapse_simpleratio(actin)
        img[:, actinch, np.newaxis, :, :, :] = actin
                
        ome_metadata = utils.construct_ome_metadata(img, img_file)
        OmeTiffWriter.save(img, output_dirpath / imgpath.name, ome_xml=ome_metadata)
    
print('Done!')